# Inteligência Artificial na era do E-Commerce: Transformando Dados em Insights

## Resumo do Projeto
Este projeto consiste no desenvolvimento de uma solução preditiva de Ciência de Dados para a empresa **Melhores Compras Ltda.**, visando estimar as vendas semanais futuras da companhia com base em dados históricos e indicadores macroeconômicos. A entrega abrange desde o diagnóstico de dados, engenharia de atributos e modelagem estatística até a justificativa técnica do algoritmo escolhido e a tradução dos resultados para a diretoria executiva (*board*).

## Contexto do Problema e Dicionário de Dados
A empresa busca prever o volume de vendas para otimizar o planejamento estratégico, financeiro e logístico. Como o faturamento é influenciado por dinâmicas de mercado, ações governamentais e sazonalidade, a modelagem deve captar múltiplos fatores externos.

O conjunto de dados fornecido (`sales.csv`) contém observações semanais compostas pelas seguintes variáveis:

* **`Date`**: Data da observação semanal (Formato original: string `DD-MM-YYYY`).
* **`Weekly_Sales`**: **Variável Alvo (*Target*)**. Faturamento de vendas na semana (em R$).
* **`Holiday_Flag`**: Variável binária ($1$ indica presença de feriado na semana; $0$ indica semana comum).
* **`Temperature`**: Temperatura média registrada na semana.
* **`Fuel_Price`**: Preço do combustível (exige divisão por $1000$ para adequação da escala em R$).
* **`CPI`**: *Consumer Price Index* — Índice de preços ao consumidor que indica o nível de atividade econômica.
* **`Unemployment`**: Taxa de desemprego (exige divisão por $1000$ para adequação da escala percentual).

## Objetivos do Notebook
1. **Saneamento e Diagnóstico:** Tratar tipos de dados, datas e adequar as escalas numéricas conforme especificação.
2. **Análise Exploratória e Engenharia de Atributos:** Identificar tendências, sazonalidades e extrair atributos temporais.
3. **Modelagem Preditiva e Validação:** Treinar algoritmos de regressão e séries temporais com validação estritamente cronológica.
4. **Interpretabilidade e Defesa Técnica:** Avaliar a relevância das variáveis e justificar tecnicamente o modelo final.

## Fase 1: Diagnóstico, Integridade e Saneamento

### Importando bibliotecas

In [1]:
# Imports

import lightgbm as lgb
import matplotlib.pyplot as plt
import missingno as msno
import numpy as np
import pandas as pd
import plotly as py
import scipy as sp
import seaborn as sns
import shap as shap
import sklearn as skl
import statsmodels as sm
import xgboost as xgb


### Carregamento da fonte de dados em um DataFrame

In [2]:
df = pd.read_csv("../data/sales.csv")

### Vizualizando a extrutura básica dos dados


In [3]:
df.head()

,Date,Weekly_Sales,Holiday_Flag,Temperature,Fuel_Price,CPI,Unemployment
0,05-02-2010,1643690.90,0,42.31,2572.0,211.096358,8106
1,12-02-2010,1641957.44,1,38.51,2548.0,211.242170,8106
2,19-02-2010,1611968.17,0,39.93,2514.0,211.289143,8106
3,26-02-2010,1409727.59,0,46.63,2561.0,211.319643,8106
4,05-03-2010,1554806.68,0,46.50,2625.0,211.350143,8106


Através da visualização confirmamos que:<br>
A coluna **"Holiday_Flag"** apresentação o número 1 para semana com feriado e 0 para quando não há.<br>
Será necessário aplicar regra de divisão por 1000 para compreensão dos dados em **"Fuel_Price"** e **"Unemployment"**.<br>

### Extrutura e dimensão do Dataframe

In [4]:
df.shape

(143, 7)

Verifica-se que temos 143 Registros/Linhas e 7 Variáveis/Colunas

### Análise Inicial dos Dados, tipos, nulos e duplicados.

In [5]:
df.info()
df.duplicated().sum()

<class 'pandas.DataFrame'>
RangeIndex: 143 entries, 0 to 142
Data columns (total 7 columns):
 #   Column        Non-Null Count  Dtype  
---  ------        --------------  -----  
 0   Date          143 non-null    str    
 1   Weekly_Sales  143 non-null    float64
 2   Holiday_Flag  143 non-null    int64  
 3   Temperature   143 non-null    float64
 4   Fuel_Price    143 non-null    float64
 5   CPI           143 non-null    float64
 6   Unemployment  143 non-null    int64  
dtypes: float64(4), int64(2), str(1)
memory usage: 7.9 KB


np.int64(0)

Ao inspecionar a base de dados enviada, observamos 143 registros semanais sem valores ausente ou duplicados.<br>
Constamos que a coluna "Date" armazena dados do tipo String e que para viabilizar futuros cálculos e ordenação cronológica para análise temporal será necessário convertê-la para datetime64.

### Estatística Descritiva Primária

In [6]:
df.describe()

,Weekly_Sales,Holiday_Flag,Temperature,Fuel_Price,CPI,Unemployment
count,1.430000e+02,143.000000,143.000000,143.000000,143.000000,143.000000
mean,1.555264e+06,0.069930,68.306783,2879.480420,215.996892,7610.419580
std,1.559808e+05,0.255926,14.250486,1068.112107,4.350890,383.748825
min,1.316899e+06,0.000000,35.400000,2.640000,210.337426,6573.000000
25%,1.458105e+06,0.000000,58.265000,2668.500000,211.549277,7348.000000
50%,1.534850e+06,0.000000,69.640000,3227.000000,215.459905,7787.000000
75%,1.614892e+06,0.000000,80.485000,3554.500000,220.531330,7838.000000
max,2.387950e+06,1.000000,91.650000,3907.000000,223.444251,8106.000000


**Diagnóstico dos Dados Brutos:**<br>

Ao rodar o describe( ), observamos:<br>
*Weekly_Sales*: varia de R$ 1.316.899,00 até R$ 2.387.950,00, com média em R$ 1.555.264,00.<br>
Não há vendas negativas nem zeradas.<br>

*Fuel_Price*: apresenta valores entre $2,64$ e $3907,00$.<br> 
Isso confirma a observação do manual da atividade sobre ajustar a escala dividindo por $1000$.<br>
Há uma hipótese de que ao menos um registro já se encontre com valor esperado após ajuste de escala $2,64$ uma vez que esse valor se enquadraria nos resultados esperados ao dividir por 1000 os valores próximos do 25% e 50%.<br>

*Unemployment*: apresenta valores entre $6573$ e $8106$. Também confirma a necessidade de dividir por $1000$ para obter o percentual real (ex: $6,573\%$ a $8,106\%$). 

### 1.2 Investigação de Anomalias de Escala (`Fuel_Price`)

Na análise descritiva anterior, o valor mínimo da variável `Fuel_Price` foi de $2.64$, enquanto o valor máximo atingiu $3907.00$.

O dicionário de dados instrui a divisão desta coluna por $1000$. Contudo, se dividirmos $2.64$ por $1000$, obteremos $\$0.00264$, um valor fora da realidade econômica para o preço de um combustível. Isso levanta a hipótese de **inconsistência de escala mista**, onde alguns registros já foram inseridos na unidade correta, enquanto a maioria foi registrada em milésimos.

**Prova Hipotética:** Para demonstrar que os valores baixos não são *outliers* ou ruídos, dividiremos a base em dois grupos usando o limiar de $1000$. Compararemos a média dos registros menores que $1000$ com a média ajustada (dividida por $1000$) dos registros maiores que $1000$. Se as médias forem estatisticamente equivalentes, comprovamos que a divergência é puramente um desvio na escala de digitação.

In [7]:
# 1. Isolando os registros onde o preço do combustível é menor que 1000 (escala em unidade)
anomalias_fuel = df[df['Fuel_Price'] < 1000]

# 2. Cálculo das médias dos dois grupos para comparação
media_base_ajustada = df[df['Fuel_Price'] >= 1000]['Fuel_Price'].mean() / 1000
media_anomala = anomalias_fuel['Fuel_Price'].mean()

# 3. Exibição dos resultados e validação
print(f"Total de registros com Fuel_Price em unidade (< 1000): {len(anomalias_fuel)}")
print(f"Média dos registros >= 1000 (ajustada por /1000): $ {media_base_ajustada:.2f}")
print(f"Média dos registros < 1000 (já em unidade):        $ {media_anomala:.2f}")

print("\n--- Registros já na escala correta ---")
anomalias_fuel[['Date', 'Weekly_Sales', 'Fuel_Price']]

Total de registros com Fuel_Price em unidade (< 1000): 15
Média dos registros >= 1000 (ajustada por /1000): $ 3.22
Média dos registros < 1000 (já em unidade):        $ 3.25

--- Registros já na escala correta ---


,Date,Weekly_Sales,Fuel_Price
6,19-03-2010,1472515.79,2.72
9,09-04-2010,1545418.53,2.77
12,30-04-2010,1425100.71,2.78
25,30-07-2010,1371986.60,2.64
36,15-10-2010,1459409.10,2.72
51,28-01-2011,1316899.31,3.01
64,29-04-2011,1455090.69,3.81
74,08-07-2011,1534849.64,3.48
103,27-01-2012,1319325.59,3.29
104,03-02-2012,1636339.65,3.36


**Conclusão da Análise de Escala:**
A comparação das médias valida a hipótese inicial:
* Média ajustada do grupo `>= 1000`: **$ 3.22**
* Média do grupo `< 1000`: **$ 3.25**

A diferença entre as médias é irrisória ($0.03$), o que prova matematicamente que os $15$ registros com valores baixos **não são outliers**, mas sim dados mantidos na escala real. Consequentemente, a correção por divisão por $1000$ deve ser aplicada exclusivamente nos valores superiores a $1000$.

### 1.3 Saneamento Definitivo da Base de Dados

Com todas as colunas diagnosticadas e a hipótese de escala mista comprovada, executamos o tratamento definitivo dos dados:

1. **Conversão Temporal e Ordenação:** Transformação da coluna `Date` para o tipo `datetime64[ns]` e reordenação cronológica da tabela com redefinição de índice (`reset_index`). Esse passo é indispensável para eliminar riscos de *Data Leakage* nas etapas de validação e modelagem.
2. **Ajuste Condicional de `Fuel_Price`:** Aplicação do operador vetorizado `np.where` para dividir por $1000$ exclusivamente os valores registrados na casa dos milhares ($\ge 1000$), preservando intactos os $15$ registros que já constavam na escala real.
3. **Ajuste Linear de `Unemployment`:** Divisão uniforme por $1000$ em toda a coluna para conversão dos valores absolutos na taxa percentual correspondente (%). Aplicação do operador `np.where` com limiares lógicos (`>= 100` para desemprego). Se a célula for executada novamente, os dados já corrigidos não sofrem nova divisão

In [9]:
# 1. Conversão temporal e ordenação cronológica
df['Date'] = pd.to_datetime(df['Date'], format='%d-%m-%Y')
df = df.sort_values(by='Date').reset_index(drop=True)

# 2. Correção de escala mista da variável Fuel_Price
df['Fuel_Price'] = np.where(df['Fuel_Price'] >= 1000, df['Fuel_Price'] / 1000, df['Fuel_Price'])

# 3. Correção de escala linear da variável Unemployment (idempotente: divide apenas se for > 100)
df['Unemployment'] = np.where(df['Unemployment'] >= 100, df['Unemployment'] / 1000, df['Unemployment'])

# 4. Verificação dos tipos e resumo estatístico pós-saneamento
print("--- Tipos de Dados Atualizados ---")
print(df.dtypes)

print("\n--- Estatística Descritiva Pós-Tratamento ---")
df.describe()

--- Tipos de Dados Atualizados ---
Date            datetime64[us]
Weekly_Sales           float64
Holiday_Flag             int64
Temperature            float64
Fuel_Price             float64
CPI                    float64
Unemployment           float64
dtype: object

--- Estatística Descritiva Pós-Tratamento ---


,Date,Weekly_Sales,Holiday_Flag,Temperature,Fuel_Price,CPI,Unemployment
count,143,1.430000e+02,143.000000,143.000000,143.000000,143.000000,143.000000
mean,2011-06-17 00:00:00,1.555264e+06,0.069930,68.306783,3.219699,215.996892,7.610420
min,2010-02-05 00:00:00,1.316899e+06,0.000000,35.400000,2.514000,210.337426,6.573000
25%,2010-10-11 12:00:00,1.458105e+06,0.000000,58.265000,2.764500,211.549277,7.348000
50%,2011-06-17 00:00:00,1.534850e+06,0.000000,69.640000,3.290000,215.459905,7.787000
75%,2012-02-20 12:00:00,1.614892e+06,0.000000,80.485000,3.594000,220.531330,7.838000
max,2012-10-26 00:00:00,2.387950e+06,1.000000,91.650000,3.907000,223.444251,8.106000
std,NaN,1.559808e+05,0.255926,14.250486,0.427313,4.350890,0.383749
